[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/07_ONNX_Runtime/01_ORT_Architecture/ORT_Architecture_Apply.ipynb)

# ORT Architecture — Apply

## Table of Contents
| # | Section | Description |
|---|---------|-------------|
| 1 | [Build and Profile](#1) | Build a model and profile ORT execution |
| 2 | [Layer Inspection](#2) | Examine EP assignment and kernel dispatch |
| 3 | [Memory Arena Experiment](#3) | Measure arena vs non-arena performance |
| 4 | [Threading Optimization](#4) | Find optimal thread configuration |
| 5 | [Batch Throughput](#5) | Measure throughput at different batch sizes |
| 6 | [Visualization](#6) | Performance charts and analysis |

In [ ]:
!pip install onnxruntime onnx numpy matplotlib -q

<a id='1'></a>
## 1. Build a CNN Model and Profile ORT Execution

We'll build a simple convolutional neural network in ONNX and use ORT's profiling to examine how the architecture layers interact during execution. The model implements:

$$Y = \text{Softmax}(W_2 \cdot \text{Relu}(\text{Pool}(\text{Conv}(X, W_1) + b_1)) + b_2)$$

This gives us Conv, Pool, Relu, MatMul, and Softmax operators — enough diversity to observe the partitioner and kernel dispatch in action.

In [ ]:
import numpy as np
import onnx
from onnx import helper, TensorProto, numpy_helper

# Build a small CNN: Conv -> Relu -> MaxPool -> Flatten -> MatMul -> Softmax
np.random.seed(42)

# Conv weights: out_channels=16, in_channels=1, kernel=3x3
W_conv = np.random.randn(16, 1, 3, 3).astype(np.float32) * 0.1
B_conv = np.zeros(16, dtype=np.float32)

# FC weights: input_features = 16*13*13 (after conv+pool on 28x28), output = 10
W_fc = np.random.randn(16 * 13 * 13, 10).astype(np.float32) * 0.1
B_fc = np.zeros(10, dtype=np.float32)

X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 1, 28, 28])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, ["batch", 10])

nodes = [
    helper.make_node("Conv", ["X", "W_conv", "B_conv"], ["conv_out"], kernel_shape=[3, 3], pads=[0,0,0,0]),
    helper.make_node("Relu", ["conv_out"], ["relu_out"]),
    helper.make_node("MaxPool", ["relu_out"], ["pool_out"], kernel_shape=[2, 2], strides=[2, 2]),
    helper.make_node("Flatten", ["pool_out"], ["flat_out"], axis=1),
    helper.make_node("MatMul", ["flat_out", "W_fc"], ["fc_out"]),
    helper.make_node("Add", ["fc_out", "B_fc"], ["logits"]),
    helper.make_node("Softmax", ["logits"], ["Y"], axis=1),
]

graph = helper.make_graph(
    nodes, "SimpleCNN", [X], [Y],
    initializer=[
        numpy_helper.from_array(W_conv, "W_conv"),
        numpy_helper.from_array(B_conv, "B_conv"),
        numpy_helper.from_array(W_fc, "W_fc"),
        numpy_helper.from_array(B_fc, "B_fc"),
    ]
)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
onnx.checker.check_model(model)
onnx.save(model, "simple_cnn.onnx")

print(f"Model built with {len(nodes)} nodes")
print(f"Parameters: Conv={W_conv.size + B_conv.size}, FC={W_fc.size + B_fc.size}")
print(f"Total parameters: {W_conv.size + B_conv.size + W_fc.size + B_fc.size:,}")

In [ ]:
import onnxruntime as ort
import time
import json

# Create profiled session
so = ort.SessionOptions()
so.enable_profiling = True
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

sess = ort.InferenceSession("simple_cnn.onnx", so, providers=["CPUExecutionProvider"])

# Run inference
x_test = np.random.randn(1, 1, 28, 28).astype(np.float32)
for _ in range(100):
    result = sess.run(None, {"X": x_test})

# Get profile
prof_file = sess.end_profiling()
print(f"Profile saved: {prof_file}")
print(f"\nOutput shape: {result[0].shape}")
print(f"Output (first sample): {result[0][0][:5]}... (softmax probabilities)")

# Parse profiling results
with open(prof_file, 'r') as f:
    prof_data = json.load(f)

print(f"\nProfiling events: {len(prof_data)}")
kernel_events = [e for e in prof_data if 'dur' in e and e.get('cat') == 'Node']
print(f"Kernel execution events: {len(kernel_events)}")

if kernel_events:
    print("\nTop operators by duration:")
    sorted_events = sorted(kernel_events, key=lambda x: x.get('dur', 0), reverse=True)[:10]
    for e in sorted_events:
        print(f"  {e.get('name', 'unknown'):30s} {e.get('dur', 0):>8} us")

<a id='2'></a>
## 2. Layer Inspection — EP Assignment and Kernel Dispatch

Let's examine how ORT's graph partitioner assigns nodes to Execution Providers and trace the execution through each architectural layer.

In [ ]:
import onnxruntime as ort

# Inspect with no optimization to see original graph structure
so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
sess_raw = ort.InferenceSession("simple_cnn.onnx", so, providers=["CPUExecutionProvider"])

# With full optimization
so_opt = ort.SessionOptions()
so_opt.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
so_opt.optimized_model_filepath = "simple_cnn_optimized.onnx"
sess_opt = ort.InferenceSession("simple_cnn.onnx", so_opt, providers=["CPUExecutionProvider"])

# Compare
print("=" * 60)
print("GRAPH STRUCTURE COMPARISON")
print("=" * 60)

model_raw = onnx.load("simple_cnn.onnx")
print(f"\nOriginal graph: {len(model_raw.graph.node)} nodes")
for n in model_raw.graph.node:
    print(f"  {n.op_type:15s} {list(n.input)} -> {list(n.output)}")

try:
    model_opt = onnx.load("simple_cnn_optimized.onnx")
    print(f"\nOptimized graph: {len(model_opt.graph.node)} nodes")
    for n in model_opt.graph.node:
        print(f"  {n.op_type:15s} {list(n.input)} -> {list(n.output)}")
    print(f"\nNode reduction: {len(model_raw.graph.node)} -> {len(model_opt.graph.node)}")
except:
    print("\n(Optimized model file not available for inspection)")

<a id='3'></a>
## 3. Memory Arena Experiment

We compare performance with and without ORT's memory arena to quantify the benefit of buffer reuse. The arena avoids per-operator `malloc/free` calls, which can be significant for graphs with many operators.

Expected saving from arena allocation:

$$\text{Overhead}_{\text{no\_arena}} \approx n_{\text{ops}} \times T_{\text{malloc}} \approx n_{\text{ops}} \times 1\text{-}10 \mu s$$

In [ ]:
import onnxruntime as ort
import numpy as np
import time

x_test = np.random.randn(32, 1, 28, 28).astype(np.float32)
n_runs = 500

# With arena (default)
so_arena = ort.SessionOptions()
so_arena.enable_cpu_mem_arena = True
so_arena.enable_mem_pattern = True
sess_arena = ort.InferenceSession("simple_cnn.onnx", so_arena, providers=["CPUExecutionProvider"])

# Without arena
so_no_arena = ort.SessionOptions()
so_no_arena.enable_cpu_mem_arena = False
so_no_arena.enable_mem_pattern = False
sess_no_arena = ort.InferenceSession("simple_cnn.onnx", so_no_arena, providers=["CPUExecutionProvider"])

# Warmup
for _ in range(50):
    sess_arena.run(None, {"X": x_test})
    sess_no_arena.run(None, {"X": x_test})

# Benchmark
arena_latencies = []
for _ in range(n_runs):
    start = time.perf_counter()
    sess_arena.run(None, {"X": x_test})
    arena_latencies.append((time.perf_counter() - start) * 1000)

no_arena_latencies = []
for _ in range(n_runs):
    start = time.perf_counter()
    sess_no_arena.run(None, {"X": x_test})
    no_arena_latencies.append((time.perf_counter() - start) * 1000)

print("Memory Arena Impact (batch=32):")
print(f"  With arena:    mean={np.mean(arena_latencies):.3f}ms, P99={np.percentile(arena_latencies, 99):.3f}ms")
print(f"  Without arena: mean={np.mean(no_arena_latencies):.3f}ms, P99={np.percentile(no_arena_latencies, 99):.3f}ms")
print(f"  Speedup: {np.mean(no_arena_latencies)/np.mean(arena_latencies):.2f}x")

<a id='4'></a>
## 4. Threading Optimization — Finding the Sweet Spot

The optimal threading configuration depends on model structure and hardware. We systematically sweep configurations to find the best latency-throughput tradeoff.

In [ ]:
import onnxruntime as ort
import numpy as np
import time
import os

x_test = np.random.randn(16, 1, 28, 28).astype(np.float32)
n_warmup = 20
n_runs = 200

# Get number of physical cores
n_cores = os.cpu_count() or 4
print(f"System CPU cores: {n_cores}")

configs = []
for intra in [1, 2, 4, min(8, n_cores), n_cores]:
    for inter in [1, 2]:
        so = ort.SessionOptions()
        so.intra_op_num_threads = intra
        so.inter_op_num_threads = inter
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        
        sess = ort.InferenceSession("simple_cnn.onnx", so, providers=["CPUExecutionProvider"])
        
        # Warmup
        for _ in range(n_warmup):
            sess.run(None, {"X": x_test})
        
        # Benchmark
        lats = []
        for _ in range(n_runs):
            start = time.perf_counter()
            sess.run(None, {"X": x_test})
            lats.append((time.perf_counter() - start) * 1000)
        
        configs.append({
            'intra': intra, 'inter': inter,
            'mean': np.mean(lats), 'p50': np.median(lats),
            'p99': np.percentile(lats, 99), 'std': np.std(lats)
        })

print(f"\n{'Intra':>6} {'Inter':>6} {'Mean(ms)':>10} {'P50(ms)':>10} {'P99(ms)':>10} {'Std(ms)':>10}")
print("-" * 55)
for c in sorted(configs, key=lambda x: x['mean']):
    print(f"{c['intra']:>6} {c['inter']:>6} {c['mean']:>10.3f} {c['p50']:>10.3f} {c['p99']:>10.3f} {c['std']:>10.3f}")

best = min(configs, key=lambda x: x['mean'])
print(f"\nBest config: intra={best['intra']}, inter={best['inter']} -> {best['mean']:.3f}ms mean latency")

<a id='5'></a>
## 5. Batch Throughput Analysis

Measuring how throughput scales with batch size reveals the compute vs memory-bound transition point:

$$\text{Throughput}(B) = \frac{B}{T(B)} \quad \text{where } T(B) = T_0 + \alpha \cdot B$$

For small $B$, overhead dominates ($T \approx T_0$), so throughput grows linearly. For large $B$, compute dominates and throughput saturates.

In [ ]:
import onnxruntime as ort
import numpy as np
import time

so = ort.SessionOptions()
so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
sess = ort.InferenceSession("simple_cnn.onnx", so, providers=["CPUExecutionProvider"])

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
throughput_results = []

for bs in batch_sizes:
    x = np.random.randn(bs, 1, 28, 28).astype(np.float32)
    
    # Warmup
    for _ in range(20):
        sess.run(None, {"X": x})
    
    # Measure
    lats = []
    for _ in range(100):
        start = time.perf_counter()
        sess.run(None, {"X": x})
        lats.append((time.perf_counter() - start) * 1000)
    
    mean_lat = np.mean(lats)
    throughput = bs / (mean_lat / 1000)  # samples/sec
    lat_per_sample = mean_lat / bs
    
    throughput_results.append({
        'batch': bs, 'latency_ms': mean_lat,
        'throughput': throughput, 'lat_per_sample': lat_per_sample
    })

print(f"{'Batch':>6} {'Latency(ms)':>12} {'Throughput':>14} {'ms/sample':>12}")
print("-" * 48)
for r in throughput_results:
    print(f"{r['batch']:>6} {r['latency_ms']:>12.3f} {r['throughput']:>12.0f}/s {r['lat_per_sample']:>10.4f}")

<a id='6'></a>
## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Batch vs Throughput
batches = [r['batch'] for r in throughput_results]
throughputs = [r['throughput'] for r in throughput_results]
axes[0,0].plot(batches, throughputs, 'go-', linewidth=2, markersize=8)
axes[0,0].set_xlabel('Batch Size')
axes[0,0].set_ylabel('Throughput (samples/sec)')
axes[0,0].set_title('Throughput vs Batch Size', fontweight='bold')
axes[0,0].set_xscale('log', base=2)
axes[0,0].grid(True, alpha=0.3)

# Plot 2: Batch vs Latency
latencies = [r['latency_ms'] for r in throughput_results]
axes[0,1].plot(batches, latencies, 'ro-', linewidth=2, markersize=8)
axes[0,1].set_xlabel('Batch Size')
axes[0,1].set_ylabel('Latency (ms)')
axes[0,1].set_title('Latency vs Batch Size', fontweight='bold')
axes[0,1].set_xscale('log', base=2)
axes[0,1].grid(True, alpha=0.3)

# Plot 3: Arena vs No-Arena
axes[1,0].hist(arena_latencies, bins=30, alpha=0.7, label='With Arena', color='#2ecc71')
axes[1,0].hist(no_arena_latencies, bins=30, alpha=0.7, label='Without Arena', color='#e74c3c')
axes[1,0].set_xlabel('Latency (ms)')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Memory Arena Impact on Latency Distribution', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Plot 4: Threading config comparison
config_labels = [f"({c['intra']},{c['inter']})" for c in configs]
config_means = [c['mean'] for c in configs]
colors = ['#3498db' if c['mean'] == best['mean'] else '#bdc3c7' for c in configs]
axes[1,1].bar(range(len(configs)), config_means, color=colors, edgecolor='black', linewidth=0.5)
axes[1,1].set_xlabel('(intra_op, inter_op) threads')
axes[1,1].set_ylabel('Mean Latency (ms)')
axes[1,1].set_title('Threading Configuration Comparison', fontweight='bold')
axes[1,1].set_xticks(range(len(configs)))
axes[1,1].set_xticklabels(config_labels, rotation=45)
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ort_architecture_apply.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cleanup
import os
for f in ['simple_cnn.onnx', 'simple_cnn_optimized.onnx']:
    if os.path.exists(f):
        os.remove(f)
for f in os.listdir('.'):
    if 'onnxruntime_profile' in f:
        os.remove(f)
print("Cleanup complete.")

## Summary

In this notebook we applied ORT architectural concepts:

1. **Built and profiled** a CNN model, examining how ORT's profiling reveals kernel-level timing
2. **Inspected layers** by comparing original vs optimized graph structure
3. **Measured arena impact** — memory arenas reduce allocation overhead significantly
4. **Optimized threading** — found the best (intra_op, inter_op) configuration empirically
5. **Analyzed throughput scaling** — demonstrated the batch size vs throughput relationship:

$$\text{Throughput}_{\text{max}} = \lim_{B \to \infty} \frac{B}{T_0 + \alpha B} = \frac{1}{\alpha}$$